### IMPORTS

In [59]:
import tensorflow as tf
import numpy as np
# import matplotlib.pyplot as plt
from tensorflow import keras
# from keras import Sequential, Input, layers, optimizers
# from keras.callbacks import EarlyStopping, ModelCheckpoint
from keras.applications.vgg16 import VGG16
from keras.applications.resnet50 import ResNet50
from keras.applications.efficientnet import EfficientNetB0
# from sklearn.metrics import classification_report, confusion_matrix
# import pandas as pd
# import seaborn as sns
# from PIL import Image

### CONFIGURATION

In [60]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
RANDOM_SEED = 42
AUTOTUNE = tf.data.AUTOTUNE

### PATHS

In [61]:
ORIGINAL_DIR = "../raw_data/Original"
AUGMENTED_DIR = "../raw_data/Augmented"

### LOAD DATASETS FROM DIRECTORY

In [62]:
## Train dataset - Augmented data

train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    AUGMENTED_DIR,
    labels="inferred",
    label_mode="binary",
    class_names=["Non-Stone", "Stone"],
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=RANDOM_SEED
)

Found 35457 files belonging to 2 classes.


In [63]:
## Full Original Dataset

full_original = tf.keras.preprocessing.image_dataset_from_directory(
    ORIGINAL_DIR,
    labels="inferred",
    label_mode="binary",
    class_names=["Non-Stone", "Stone"],
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    seed=RANDOM_SEED
)

Found 3364 files belonging to 2 classes.


In [64]:
full_original_shuffled = full_original.shuffle(
    buffer_size=len(full_original) * BATCH_SIZE,
    seed=RANDOM_SEED,
    reshuffle_each_iteration=False
)

### SPLIT ORIGINAL DATASET

In [65]:
## Val and Test dataset - From Full Original dataset

total_batches = len(full_original)
val_batches = total_batches // 2

val_dataset = full_original_shuffled.take(val_batches).cache().prefetch(AUTOTUNE)
test_dataset = full_original_shuffled.skip(val_batches).cache().prefetch(AUTOTUNE)

print(train_dataset.class_names)
print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))

['Non-Stone', 'Stone']
1109
53
53


### LABEL DISTRIBUTION

In [66]:
true_val_labels = np.concatenate([y.numpy() for _, y in val_dataset]).flatten().astype(int)
print(f"Val unique labels : {np.unique(true_val_labels)}")
print(f"Val label counts  : {np.bincount(true_val_labels)}")

Val unique labels : [0 1]
Val label counts  : [891 805]


2026-03-11 21:31:37.331617: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


### PREPROCESSING LAYERS

In [67]:
vgg_preprocess = tf.keras.applications.vgg16.preprocess_input
resnet_preprocess = tf.keras.applications.resnet50.preprocess_input
efficientnet_preprocess = tf.keras.applications.efficientnet.preprocess_input

In [68]:
def prepare(dataset, preprocess_input):
    return (
        dataset.map(lambda img, label: (preprocess_input(img), label), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
    )

In [69]:
def get_datasets(preprocess_input):
    return (
        prepare(train_dataset, preprocess_input),
        prepare(val_dataset, preprocess_input),
        prepare(test_dataset, preprocess_input)
    )